# Clarksons VLCC cargo counts (by decade × load region)

Source: `broker_research/clarksons/CARGO COUNTS DATA.xlsx` (2022–2026).

- **Demand side** of Tool B: cargoes per load decade
- 2026 splits **MEG/STS WCI** vs **RSEA** — use `meg_plus_rsea` for y/y vs older MEG
- Compare to Kpler MEG VLCC monthly fixtures
- Ships-per-cargo: evening PDF `MEG_SHIPS_EX_SUBS` when available, else MEG position-list proxy
- Seasonal Jan–Dec overlay with region × decade toggles (section E)

In [1]:
from datetime import date
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "packages" / "clarksons_ingest"))

from freight import kpler as kp
from freight.clarksons_cargo_counts import (
    cargoes_wide_by_region,
    load_cargo_counts,
    meg_cargoes_comparable,
    plot_seasonal_cargoes,
    seasonal_cargoes,
    ships_from_evening_pdf,
    ships_per_cargo,
    ships_proxy_from_meg_list,
)
from freight.clarksons_meg_list import load_all_meg_lists

pd.options.display.float_format = lambda x: f"{x:,.2f}"
CACHE = ROOT / "data" / "cache"
CLARK = ROOT / "broker_research" / "clarksons"
CARGO_XLSX = CLARK / "CARGO COUNTS DATA.xlsx"
print("ROOT", ROOT)

ROOT c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\freight_dashboard


## A. Load + MEG / RSEA

In [2]:
cargo = load_cargo_counts(CARGO_XLSX)
CACHE.mkdir(parents=True, exist_ok=True)
cargo.to_parquet(CACHE / "clarksons_cargo_counts.parquet", index=False)
n_relabel = int(cargo["year_relabeled"].sum()) if "year_relabeled" in cargo.columns else 0
display(Markdown(
    f"Loaded **{len(cargo)}** rows, years {sorted(cargo['year'].unique())}, "
    f"regions: {sorted(cargo.loc[~cargo['is_aggregate'], 'region'].unique())}. "
    f"Year-relabeled cells (2025 tab dated 2024): **{n_relabel}**."
))

meg = meg_cargoes_comparable(cargo)
meg_ttl = meg.loc[meg["decade"] == "TTL"].copy()
display(meg_ttl.tail(12))

fig = go.Figure()
fig.add_trace(go.Bar(x=meg_ttl["month"], y=meg_ttl["meg"], name="MEG (ex-RSEA in 2026)"))
fig.add_trace(go.Bar(x=meg_ttl["month"], y=meg_ttl["rsea"], name="RSEA"))
fig.update_layout(
    barmode="stack",
    title="MEG + Red Sea VLCC cargoes (monthly TTL)",
    height=420,
    yaxis_title="cargoes",
)
fig.show()

# Decade bars for latest complete-ish months
dec = meg.loc[meg["decade"] != "TTL"].copy()
dec["ym"] = dec["month"].dt.strftime("%Y-%m")
recent = sorted(dec["ym"].unique())[-8:]
drec = dec.loc[dec["ym"].isin(recent)]
fig_d = go.Figure()
for decade, color in [("1st", "#1f77b4"), ("2nd", "#ff7f0e"), ("3rd", "#2ca02c")]:
    sub = drec.loc[drec["decade"] == decade]
    fig_d.add_trace(go.Bar(x=sub["month"], y=sub["meg_plus_rsea"], name=decade, marker_color=color))
fig_d.update_layout(barmode="group", title="MEG+RSEA cargoes by decade (recent)", height=400)
fig_d.show()

Loaded **2352** rows, years [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)], regions: ['CARIBS_ECMEX', 'ECSA', 'MED', 'MEG', 'RSEA', 'UKC', 'USG', 'WAF', 'WCSA']. Year-relabeled cells (2025 tab dated 2024): **0**.

,month,decade,meg,rsea,meg_plus_rsea,year
195,2026-01-01,TTL,130.00,0.00,130.00,2026
199,2026-02-01,TTL,134.00,0.00,134.00,2026
203,2026-03-01,TTL,49.00,41.00,90.00,2026
207,2026-04-01,TTL,46.00,27.00,73.00,2026
211,2026-05-01,TTL,49.00,30.00,79.00,2026
215,2026-06-01,TTL,78.00,30.00,108.00,2026
219,2026-07-01,TTL,92.00,39.00,131.00,2026
223,2026-08-01,TTL,114.00,34.00,148.00,2026
227,2026-09-01,TTL,127.00,20.00,147.00,2026
231,2026-10-01,TTL,83.00,7.00,90.00,2026


## B. Load-region mix (monthly TTL)

In [3]:
wide = cargoes_wide_by_region(
    cargo,
    decade="TTL",
    regions=["MEG", "RSEA", "WAF", "USG", "ECSA", "WCSA", "CARIBS_ECMEX", "UKC", "MED"],
)
display(wide.tail(8))

fig_r = go.Figure()
for col in [c for c in wide.columns if c != "month"]:
    fig_r.add_trace(go.Scatter(x=wide["month"], y=wide[col], name=col, mode="lines"))
fig_r.update_layout(title="VLCC cargoes by load region (TTL)", height=450, yaxis_title="cargoes")
fig_r.show()

# Atlantic share vs MEG+RSEA
mix = wide.copy()
atl_cols = [c for c in ("WAF", "USG", "ECSA", "WCSA", "CARIBS_ECMEX", "UKC", "MED") if c in mix.columns]
mix["atl"] = mix[atl_cols].sum(axis=1, min_count=1)
mix["meg_rsea"] = mix.get("MEG", 0).fillna(0) + mix.get("RSEA", 0).fillna(0)
mix["atl_share"] = mix["atl"] / (mix["atl"] + mix["meg_rsea"]).replace(0, pd.NA)
fig_s = go.Figure(go.Scatter(x=mix["month"], y=mix["atl_share"], mode="lines+markers", name="Atlantic share"))
fig_s.update_layout(title="Atlantic load share vs MEG+RSEA", height=360, yaxis_tickformat=".0%")
fig_s.show()

region,month,CARIBS_ECMEX,ECSA,MED,MEG,RSEA,UKC,USG,WAF,WCSA
52,2026-05-01,17.00,29.00,1.00,49.00,30.00,0.00,46.00,22.00,4.00
53,2026-06-01,13.00,31.00,1.00,78.00,30.00,0.00,39.00,19.00,4.00
54,2026-07-01,7.00,30.00,5.00,92.00,39.00,0.00,19.00,13.00,3.00
55,2026-08-01,8.00,35.00,22.00,114.00,34.00,1.00,14.00,22.00,9.00
56,2026-09-01,4.00,33.00,8.00,127.00,20.00,1.00,31.00,16.00,3.00
57,2026-10-01,2.00,31.00,9.00,83.00,7.00,0.00,21.00,8.00,5.00
58,2026-11-01,0.00,7.00,0.00,0.00,4.00,0.00,6.00,5.00,0.00
59,2026-12-01,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


## C. Kpler MEG fixtures vs Clarksons MEG+RSEA cargoes

Not 1:1 (fixtures ≠ liftings; destinations differ; timing). Directional check only.

In [4]:
config = kp.liquids_config(ROOT)
fx_start = date(2024, 1, 1)
fx = kp.fixtures_vlcc_monthly(config, fx_start, date.today(), from_zones=[kp.MEG_ZONE], to_zones=None)
cmp = meg_ttl.loc[meg_ttl["month"] >= "2024-01-01", ["month", "meg_plus_rsea"]].merge(
    fx.rename(columns={"Date": "month", "n_fixtures": "kpler_meg_fixtures"}),
    on="month",
    how="outer",
).sort_values("month")
display(cmp.tail(12))

fig_k = make_subplots(specs=[[{"secondary_y": True}]])
fig_k.add_trace(
    go.Bar(x=cmp["month"], y=cmp["meg_plus_rsea"], name="Clarksons MEG+RSEA cargoes"),
    secondary_y=False,
)
fig_k.add_trace(
    go.Scatter(x=cmp["month"], y=cmp["kpler_meg_fixtures"], name="Kpler MEG VLCC fixtures", mode="lines+markers"),
    secondary_y=True,
)
fig_k.update_layout(title="Clarksons cargoes vs Kpler MEG fixtures (monthly)", height=420)
fig_k.update_yaxes(title_text="Clarksons cargoes", secondary_y=False)
fig_k.update_yaxes(title_text="Kpler fixtures", secondary_y=True)
fig_k.show()

,month,meg_plus_rsea,kpler_meg_fixtures
24,2026-01-01,130.00,NaN
25,2026-02-01,134.00,274.00
26,2026-03-01,90.00,235.00
27,2026-04-01,73.00,82.00
28,2026-05-01,79.00,21.00
29,2026-06-01,108.00,72.00
30,2026-07-01,131.00,120.00
31,2026-08-01,148.00,70.00
32,2026-09-01,147.00,105.00
33,2026-10-01,90.00,21.00


## D. Ships per cargo

Prefer evening PDF `MEG_SHIPS_EX_SUBS`. Fallback: mean MEG position-list size in that decade (proxy — not ex-subs).

In [5]:
# Base = list proxy; overlay evening PDF ships-ex-subs where parsed (PDF wins).
meg_list = load_all_meg_lists(CLARK)
ships = ships_proxy_from_meg_list(meg_list)
pdf_parts = []
for pdf in sorted(set(CLARK.glob("*Evening*.pdf")) | set(CLARK.glob("*evening*.pdf"))):
    try:
        part = ships_from_evening_pdf(pdf)
        if not part.empty:
            pdf_parts.append(part)
            print("parsed ships", pdf.name, len(part))
    except Exception as exc:  # noqa: BLE001
        print("skip", pdf.name, type(exc).__name__, exc)
if pdf_parts:
    pdf_ships = pd.concat(pdf_parts, ignore_index=True).drop_duplicates(["month", "decade"], keep="last")
    ships = pd.concat([ships, pdf_ships], ignore_index=True).drop_duplicates(["month", "decade"], keep="last")

spc = ships_per_cargo(meg, ships, cargo_col="meg_plus_rsea", ships_col="ships")
spc.to_parquet(CACHE / "clarksons_ships_per_cargo.parquet", index=False)
src_counts = ships.groupby("ships_source").size().to_dict() if "ships_source" in ships.columns else {}
display(Markdown(f"Ships rows by source: `{src_counts}`. SPC rows: **{len(spc)}**."))
display(spc.tail(12))

if not spc.empty:
    fig_p = go.Figure()
    for decade in ["1st", "2nd", "3rd"]:
        sub = spc.loc[spc["decade"] == decade]
        fig_p.add_trace(go.Scatter(x=sub["month"], y=sub["ships_per_cargo"], name=decade, mode="lines+markers"))
    fig_p.update_layout(
        title="MEG ships per cargo by decade (PDF ex-subs overlays list proxy)",
        height=400,
        yaxis_title="ships / cargo",
    )
    fig_p.show()

parsed ships VLCC Evening Report 05 10 2026.pdf 3


Ships rows by source: `{'evening_pdf': 3, 'meg_list_proxy': 3}`. SPC rows: **6**.

,month,decade,meg,rsea,meg_plus_rsea,year,ships,ships_per_cargo
0,2026-09-01,3rd,47.00,5.00,52.00,2026,177.25,3.41
1,2026-10-01,1st,42.00,2.00,44.00,2026,187.14,4.25
2,2026-10-01,2nd,30.00,4.00,34.00,2026,178.00,5.24
3,2026-10-01,3rd,11.00,1.00,12.00,2026,93.00,7.75
4,2026-11-01,1st,0.00,0.00,0.00,2026,61.00,<NA>
5,2026-11-01,2nd,0.00,4.00,4.00,2026,52.00,13.00


## E. Seasonal demand (toggle region × decade)

Jan–Dec overlay across years (latest year highlighted). Use the dropdowns to switch **region** and **decade**.

- `MEG+RSEA` is the y/y-comparable Gulf series (RSEA folded in from 2026).
- Trailing zeros in the current year are masked so incomplete book months do not plot as zero demand.

In [6]:
# Defaults if ipywidgets is unavailable — edit these two lines.
SEASON_REGION = "MEG+RSEA"
SEASON_DECADE = "TTL"

regions = ["MEG+RSEA"] + sorted(
    r for r in cargo.loc[~cargo["is_aggregate"], "region"].unique()
)
decades = ["TTL", "1st", "2nd", "3rd"]


def _show_seasonal(region: str, decade: str):
    seas = seasonal_cargoes(cargo, region=region, decade=decade)
    fig = plot_seasonal_cargoes(seas)
    display(Markdown(
        f"**{region}** · **{decade}** — years {sorted(seas['year'].unique()) if len(seas) else []} "
        f"({len(seas)} month-points; trailing zeros in latest year masked)."
    ))
    fig.show()
    if len(seas):
        display(
            seas.pivot_table(index="cal_month", columns="year", values="cargoes")
            .reindex(range(1, 13))
            .rename(index={i + 1: m for i, m in enumerate(
                ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
            )})
        )


try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    region_dd = widgets.Dropdown(options=regions, value=SEASON_REGION, description="Region:")
    decade_dd = widgets.Dropdown(options=decades, value=SEASON_DECADE, description="Decade:")
    out = widgets.Output()

    def _on_change(_=None):
        with out:
            clear_output(wait=True)
            _show_seasonal(region_dd.value, decade_dd.value)

    region_dd.observe(_on_change, names="value")
    decade_dd.observe(_on_change, names="value")
    display(widgets.HBox([region_dd, decade_dd]), out)
    _on_change()
except ImportError:
    display(Markdown(
        f"`ipywidgets` not installed — showing **{SEASON_REGION}** / **{SEASON_DECADE}**. "
        "Edit `SEASON_REGION` / `SEASON_DECADE` above, or `pip install ipywidgets`."
    ))
    _show_seasonal(SEASON_REGION, SEASON_DECADE)

Output()